In [ ]:
/_static/db/steenberg.db

# Detective: de valse factuur

:::{admonition} Leerdoelen
:class: tip

In deze les los je een fraudeonderzoek op met SQL. Nieuwe leerstof is er niet: je combineert alles wat je al kan.

Na deze les kan je:
- een onbekende database verkennen aan de hand van een schema
- met SELECT, WHERE, LIKE en JOIN sporen zoeken in meerdere tabellen
- resultaten van verschillende queries combineren tot één sluitende conclusie
:::

## Het dossier

Bij **Bouwgroothandel Steenberg nv** in Aalst is iets aan de hand. Bij de maandelijkse controle van de facturatie ontdekte de interne audit dat er met een factuur geknoeid werd: iemand verlaagde de prijzen nadat de factuur al opgemaakt was. Niemand keurde die kortingen goed.

De directie wil weten wie erachter zit — en jij leidt het onderzoek, samen met je duo-partner. Alles wat je nodig hebt zit in de bedrijfsdatabase: het verkoopsysteem, de badge-registratie aan de personeelsingang, de metadata van de mailserver en de verslagen van de verhoren die intussen werden afgenomen.

## De database

- **medewerkers**(medewerker_id, voornaam, achternaam, functie, afdeling)
- **accounts**(account_id, medewerker_id, gebruikersnaam) — de logins voor het verkoopsysteem
- **klanten**(klant_id, bedrijfsnaam, contactpersoon, email, gemeente)
- **producten**(product_id, naam, categorie, eenheidsprijs)
- **facturen**(factuur_id, klant_id, datum, bedrag)
- **prijswijzigingen**(wijziging_id, factuur_id, product_id, oude_prijs, nieuwe_prijs, datum, tijdstip, account_id) — het logboek van het verkoopsysteem
- **badge_logs**(log_id, medewerker_id, datum, tijdstip, richting) — `'in'` of `'uit'` aan de personeelsingang
- **verhoren**(verhoor_id, medewerker_id, datum, tekst)
- **emails**(email_id, afzender, ontvanger, onderwerp, datum)
- **auditverslagen**(verslag_id, datum, categorie, tekst)
- **controle**(verdachte, uitkomst) — hier controleer je op het einde je antwoord

:::{admonition} Spelregels
:class: warning
- Werk in duo's: de ene typt, de andere denkt mee en noteert wat jullie vinden.
- Elke stap levert een nieuw spoor op: een nummer, een naam, een tijdstip. **Schrijf die op**, je hebt ze in de volgende stappen nodig.
- Zit je vast? Onder elke stap staat een hint. Klap die pas open als je het echt zelf geprobeerd hebt.
- En uiteraard: een echte detective kijkt niet stiekem in de tabel `controle`. Die is alleen voor je eindantwoord.
:::

## Stap 1: het auditverslag

Alles begint bij de interne audit. Het verslag dat de fraude aan het licht bracht, werd geschreven op **2 juni 2025**. Vraag het op — en lees het aandachtig.

In [ ]:
--- Stap 1: vraag het auditverslag van 2 juni 2025 op.

:::{admonition} Hint bij stap 1
:class: tip dropdown
Filter de tabel `auditverslagen` op datum. Datums zijn tekst in het formaat `YYYY-MM-DD`.
``` SQL
SELECT tekst
FROM auditverslagen
WHERE datum = '...';
```
:::

## Stap 2: wie is de klant?

Het verslag noemt een factuurnummer. Zoek op voor welke klant die factuur werd opgemaakt. Je hebt daarvoor twee tabellen nodig: `facturen` en `klanten`.

Noteer de bedrijfsnaam **én de contactpersoon**.

In [ ]:
--- Stap 2: zoek de klant (en de contactpersoon) van de verdachte factuur.

:::{admonition} Hint bij stap 2
:class: tip dropdown
Koppel de twee tabellen via `klant_id` en filter op het factuurnummer uit het verslag.
``` SQL
SELECT k.bedrijfsnaam, k.contactpersoon, k.gemeente, f.datum
FROM facturen f
JOIN klanten k ON f.klant_id = k.klant_id
WHERE f.factuur_id = ...;
```
:::

## Stap 3: het logboek

Volgens de audit werden de prijzen aangepast nadat de factuur al was opgemaakt. Het verkoopsysteem houdt elke prijswijziging bij in de tabel `prijswijzigingen`.

Zoek alle wijzigingen op de verdachte factuur. Kijk goed naar de datum, het tijdstip en het account. Wat valt je op?

*Extra: met een JOIN op `producten` zie je ook wélke producten plots spotgoedkoop werden.*

In [ ]:
--- Stap 3: bekijk alle prijswijzigingen op de verdachte factuur.

:::{admonition} Hint bij stap 3
:class: tip dropdown
``` SQL
SELECT p.naam, w.oude_prijs, w.nieuwe_prijs, w.datum, w.tijdstip, w.account_id
FROM prijswijzigingen w
JOIN producten p ON w.product_id = p.product_id
WHERE w.factuur_id = ...;
```
Vergelijk de tijdstippen met gewone kantooruren. Noteer het `account_id`.
:::

## Stap 4: van account naar naam

De wijzigingen gebeurden met een account, niet met een naam. Zoek uit van wie dat account is. Daarvoor koppel je `accounts` aan `medewerkers`.

Noteer ook de `medewerker_id`: die heb je in de volgende stap nodig.

In [ ]:
--- Stap 4: van wie is het account waarmee de prijzen werden aangepast?

:::{admonition} Hint bij stap 4
:class: tip dropdown
``` SQL
SELECT m.medewerker_id, m.voornaam, m.achternaam, m.functie, m.afdeling
FROM accounts a
JOIN medewerkers m ON a.medewerker_id = m.medewerker_id
WHERE a.account_id = ...;
```
:::

## Stap 5: het alibi

Vreemd. De eigenaar van het account werkt op de boekhouding — en ontkent in alle toonaarden. Lees eerst haar verklaring in de tabel `verhoren` (zoek op haar `medewerker_id`).

Klopt haar verhaal? Er is één manier om dat na te gaan: de badge-registratie aan de personeelsingang. Controleer haar `badge_logs` op de dag van de fraude.

In [ ]:
--- Stap 5a: lees het verhoor van de eigenaar van het account.

--- Stap 5b: controleer haar badge_logs op de dag van de fraude. Klopt haar alibi?

:::{admonition} Hint bij stap 5
:class: tip dropdown
Twee aparte queries:
``` SQL
SELECT tekst
FROM verhoren
WHERE medewerker_id = ...;
```
``` SQL
SELECT tijdstip, richting
FROM badge_logs
WHERE medewerker_id = ... AND datum = '...'
ORDER BY tijdstip;
```
Vergelijk haar laatste badge-event met het tijdstip van de prijswijzigingen.
:::

## Stap 6: wie was er die avond?

Haar alibi klopt: ze was al uren buiten toen de prijzen werden aangepast. Iemand anders gebruikte dus haar login — en die persoon moet die avond **in het gebouw** geweest zijn.

Zoek alle badge-events van de fraudedag na 18:00 en zet er meteen de namen bij. Wie badgede er 's avonds nog *binnen*? En wat zegt die persoon zelf in zijn verhoor?

In [ ]:
--- Stap 6a: alle badge-events van de fraudedag na 18:00, met naam en functie.

--- Stap 6b: lees het verhoor van je verdachte. Spreekt het de badge-gegevens tegen?

:::{admonition} Hint bij stap 6
:class: tip dropdown
``` SQL
SELECT b.tijdstip, b.richting, m.medewerker_id, m.voornaam, m.achternaam, m.functie
FROM badge_logs b
JOIN medewerkers m ON b.medewerker_id = m.medewerker_id
WHERE b.datum = '...' AND b.tijdstip > '18:00'
ORDER BY b.tijdstip;
```
Let op de kolom `richting`: wie ging er alleen maar buiten, en wie kwam er nog binnen?
:::

## Stap 7: het motief

Je hebt een verdachte: iemand die 's avonds laat binnen badgede, maar beweert dat hij thuis was. Alleen: waarom zou hij dit doen?

Kijk nog eens naar de contactpersoon van de klant uit stap 2. Valt je iets op aan die naam?

Doorzoek daarna de metadata van de mailserver: welke e-mails werden verstuurd van of naar iemand bij die klant? Gebruik `LIKE` op het e-maildomein.

In [ ]:
--- Stap 7: zoek alle e-mails van of naar het bedrijf van de klant.

:::{admonition} Hint bij stap 7
:class: tip dropdown
Het e-maildomein vind je in de kolom `email` van de tabel `klanten`.
``` SQL
SELECT afzender, ontvanger, onderwerp, datum
FROM emails
WHERE afzender LIKE '%...%' OR ontvanger LIKE '%...%';
```
:::

## De ontknoping

Je hebt nu alles: de **gelegenheid** (de badge-registratie), de **middelen** (een gestolen wachtwoord — herlees het verhoor uit stap 5) en het **motief** (de e-mails en de familieband).

Wijs je dader aan in de tabel `controle`. Vul de volledige naam in, precies zoals die in `medewerkers` staat:

``` SQL
SELECT uitkomst
FROM controle
WHERE verdachte = 'Voornaam Achternaam';
```

In [ ]:
--- Wie pleegde de fraude? Controleer je antwoord in de tabel controle.

:::{admonition} Zaak gesloten?
:class: tip
Kreeg je *"Juist!"* te zien? Proficiat, detective. Overloop dan samen nog even het dossier:

- Welke query leverde het doorslaggevende bewijs?
- De fraudeur gebruikte het account van iemand anders. Hoe raakte hij aan dat wachtwoord — en welke afspraak binnen het bedrijf had dit kunnen voorkomen?
- Stel dat de fraudeur zijn sporen had willen wissen: in welke tabellen had hij dan moeten knoeien?
:::